In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client["MercadoLibre"] 
coleccion = db["e-commerce"] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [2]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza completada.")

NOMBRE_GRUPO = "e-commerce"
datos_finales = []
driver = None

options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

try:
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://listado.mercadolibre.cl/notebook")

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")
        time.sleep(5)

        # ESPERA EXPLÍCITA (Corregida la sintaxis de paréntesis aquí)
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "poly-card"))
        )

        bloques = driver.find_elements(By.CLASS_NAME, "poly-card")

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.CLASS_NAME, "poly-component__title").text
                marca = bloque.find_element(By.CLASS_NAME, "poly-component__seller").text
                precio = bloque.find_element(By.CLASS_NAME, "poly-price__current").text
                
                try:
                    calificacion = bloque.find_element(By.CLASS_NAME, "poly-component__review-compacted").text
                except:
                    calificacion = "0"

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "marca": marca,
                    "calificacion": calificacion,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                })
            except:
                continue

        # AVANZAR PÁGINA
        try:
            boton_siguiente = driver.find_element(By.CSS_SELECTOR, "a.andes-pagination__link[title='Siguiente']")
            driver.execute_script("arguments[0].scrollIntoView();", boton_siguiente)
            time.sleep(1) # Respiro para el scroll
            driver.execute_script("arguments[0].click();", boton_siguiente)
            print("Cambiando de página...")
        except:
            print("Llegamos al final o el botón no es visible.")
            break

    print(f"Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print(f"Error en Selenium: {e}")

finally:
    if driver:
        driver.quit()

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["MercadoLibre"]
    coleccion = db["e-commerce"]

    if datos_finales:
        for d in datos_finales:
            v_sucio = str(d.get("valor", "0"))
            # Limpiamos el precio de verdad
            v_limpio = v_sucio.replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
            try:
                # Si el precio tiene algo como "7% OFF", nos quedamos solo con los números iniciales
                import re
                solo_numeros = re.findall(r'\d+', v_limpio)
                if solo_numeros:
                    d["valor"] = float(solo_numeros[0])
                else:
                    d["valor"] = 0.0
            except:
                d["valor"] = 0.0

        coleccion.insert_many(datos_finales)
        print("Datos cargados en MongoDB correctamente.")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")
    

🧹 Limpieza completada.
Navegador iniciado.
--- Procesando Página 1 ---
Cambiando de página...
--- Procesando Página 2 ---
Cambiando de página...
--- Procesando Página 3 ---
Cambiando de página...
Extracción terminada: 143 productos.
Datos cargados en MongoDB correctamente.
